In [1]:
import json
import os
import re
from typing import Any, Callable, Dict

import requests
from dotenv import load_dotenv
from google import genai
from google.genai import types

MODEL_NAME = "gemini-2.5-flash-lite"
BINANCE_TICKER_PRICE_URL = "https://api.binance.com/api/v3/ticker/price"
REQUEST_TIMEOUT_SECONDS = 15

class ConfigurationError(Exception):
    """Raised when required configuration is missing or invalid."""

class ExternalAPIError(Exception):
    """Raised when an external API call fails."""


Ensure that you created a Gemini API Key and added it to the `.env` file before executing the next command

In [2]:
def load_and_validate_env() -> str:
    """
    Load environment variables and validate the Gemini API key.
    """
    load_dotenv()

    api_key = os.getenv("GEMINI_API_KEY", "").strip()
    if not api_key or api_key.startswith("ADD YOUR"):
        raise ConfigurationError(
            "GEMINI_API_KEY not found or invalid in .env file."
        )

    return api_key


def create_gemini_client(api_key: str) -> genai.Client:
    """
    Create and return a Gemini client.
    """
    try:
        return genai.Client(api_key=api_key)
    except Exception as exc:
        raise ConfigurationError(
            f"Failed to initialize Gemini client: {exc}"
        ) from exc


def validate_symbol(symbol: str) -> str:
    """
    Validate and normalize a Binance trading pair symbol.

    Accepts values like 'BTCUSDT' and returns uppercase normalized symbol.
    """
    if not isinstance(symbol, str):
        raise ValueError("Symbol must be a string.")

    normalized = symbol.strip().upper()

    if not normalized:
        raise ValueError("Symbol cannot be empty.")

    if not re.fullmatch(r"[A-Z0-9]{5,20}", normalized):
        raise ValueError(
            "Invalid symbol format. Expected something like BTCUSDT or ETHUSDT."
        )

    return normalized


def get_crypto_price(symbol: str) -> Dict[str, Any]:
    """
    Get the current price of a cryptocurrency from Binance Spot API.

    Returns a normalized dictionary suitable for tool responses.
    """
    normalized_symbol = validate_symbol(symbol)

    try:
        response = requests.get(
            BINANCE_TICKER_PRICE_URL,
            params={"symbol": normalized_symbol},
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
    except requests.Timeout as exc:
        raise ExternalAPIError(
            f"Timeout while calling Binance for symbol {normalized_symbol}."
        ) from exc
    except requests.RequestException as exc:
        raise ExternalAPIError(
            f"HTTP error while calling Binance for symbol {normalized_symbol}: {exc}"
        ) from exc

    try:
        data = response.json()
    except ValueError as exc:
        raise ExternalAPIError(
            f"Binance returned invalid JSON for symbol {normalized_symbol}."
        ) from exc

    if not isinstance(data, dict):
        raise ExternalAPIError(
            f"Unexpected Binance response type for symbol {normalized_symbol}: {type(data).__name__}"
        )

    if "code" in data and "msg" in data:
        raise ExternalAPIError(
            f"Binance API error for {normalized_symbol}: {data['code']} - {data['msg']}"
        )

    returned_symbol = data.get("symbol")
    returned_price = data.get("price")

    if not returned_symbol or returned_price is None:
        raise ExternalAPIError(
            f"Unexpected Binance response payload for {normalized_symbol}: {data}"
        )

    try:
        numeric_price = float(returned_price)
    except (TypeError, ValueError) as exc:
        raise ExternalAPIError(
            f"Invalid price format returned by Binance for {normalized_symbol}: {returned_price}"
        ) from exc

    return {
        "symbol": returned_symbol,
        "price": numeric_price,
        "currency": "USDT",
        "source": "Binance Spot API",
    }


def build_tools() -> list[types.Tool]:
    """
    Build Gemini function-calling tool declarations.
    """
    return [
        types.Tool(
            function_declarations=[
                types.FunctionDeclaration(
                    name="get_crypto_price",
                    description="Get cryptocurrency price in USDT from Binance.",
                    parameters={
                        "type": "object",
                        "properties": {
                            "symbol": {
                                "type": "string",
                                "description": (
                                    "The cryptocurrency trading pair symbol, "
                                    "for example BTCUSDT or ETHUSDT. "
                                    "Bitcoin uses BTCUSDT. Ethereum uses ETHUSDT."
                                ),
                            }
                        },
                        "required": ["symbol"],
                    },
                )
            ]
        )
    ]


def extract_function_calls(response: Any) -> list[Any]:
    """
    Extract function calls from a Gemini response safely.
    """
    function_calls = []

    candidates = getattr(response, "candidates", None) or []
    for candidate in candidates:
        content = getattr(candidate, "content", None)
        parts = getattr(content, "parts", None) or []

        for part in parts:
            function_call = getattr(part, "function_call", None)
            if function_call:
                function_calls.append(function_call)

    return function_calls


def run_tool_calls(
    function_calls: list[Any],
    available_functions: Dict[str, Callable[..., Dict[str, Any]]],
) -> list[types.Part]:
    """
    Execute tool calls requested by the model and convert outputs into Gemini tool response parts.
    """
    tool_response_parts: list[types.Part] = []

    for function_call in function_calls:
        function_name = function_call.name
        function_args = dict(function_call.args or {})

        if function_name not in available_functions:
            raise ValueError(f"Unknown function requested by model: {function_name}")

        try:
            result = available_functions[function_name](**function_args)
            payload = {"ok": True, "result": result}
        except Exception as exc:
            payload = {
                "ok": False,
                "error": str(exc),
            }

        tool_response_parts.append(
            types.Part.from_function_response(
                name=function_name,
                response=payload,
            )
        )

    return tool_response_parts


def ask_model_with_tools(
    client: genai.Client,
    prompt: str,
    model_name: str = MODEL_NAME,
) -> str:
    """
    Send a prompt to Gemini, allow tool use, execute requested tools,
    and return the final natural-language answer.
    """
    tools = build_tools()

    available_functions: Dict[str, Callable[..., Dict[str, Any]]] = {
        "get_crypto_price": get_crypto_price,
    }

    try:
        first_response = client.models.generate_content(
            model=model_name,
            contents=prompt,
            config=types.GenerateContentConfig(
                tools=tools,
            ),
        )
    except Exception as exc:
        raise RuntimeError(f"Initial Gemini request failed: {exc}") from exc

    function_calls = extract_function_calls(first_response)

    if not function_calls:
        text = getattr(first_response, "text", None)
        if text:
            return text.strip()
        raise RuntimeError("Model returned no text and no function call.")

    tool_response_parts = run_tool_calls(function_calls, available_functions)

    try:
        final_response = client.models.generate_content(
            model=model_name,
            contents=[
                types.Content(
                    role="user",
                    parts=[types.Part.from_text(text=prompt)],
                ),
                first_response.candidates[0].content,
                types.Content(
                    role="tool",
                    parts=tool_response_parts,
                ),
            ],
        )
    except Exception as exc:
        raise RuntimeError(f"Final Gemini request failed: {exc}") from exc

    text = getattr(final_response, "text", None)
    if text:
        return text.strip()

    try:
        return json.dumps(final_response.model_dump(), indent=2, ensure_ascii=False)
    except Exception:
        return str(final_response)

In [3]:
def main() -> None:
    """
    Main execution entrypoint.
    """
    try:
        api_key = load_and_validate_env()
        client = create_gemini_client(api_key)

        print("Environment loaded successfully.")
        print(f"Gemini client initialized successfully. Model: {MODEL_NAME}")

        prompt = "What is the current price of Bitcoin?"
        answer = ask_model_with_tools(client=client, prompt=prompt, model_name=MODEL_NAME)

        print("\nFinal response:")
        print(answer)

        direct_price = get_crypto_price("BTCUSDT")
        print("\nDirect Binance check:")
        print(direct_price)

    except ConfigurationError as exc:
        print(f"[CONFIG ERROR] {exc}")
    except ExternalAPIError as exc:
        print(f"[EXTERNAL API ERROR] {exc}")
    except ValueError as exc:
        print(f"[VALIDATION ERROR] {exc}")
    except Exception as exc:
        print(f"[UNEXPECTED ERROR] {exc}")

In [4]:
if __name__ == "__main__":
    main()

Environment loaded successfully.
Gemini client initialized successfully. Model: gemini-2.5-flash-lite

Final response:
The current price of Bitcoin (BTC) is **$69,913.38** against Tether (USDT), according to the Binance Spot API.

Direct Binance check:
{'symbol': 'BTCUSDT', 'price': 69898.7, 'currency': 'USDT', 'source': 'Binance Spot API'}
